<a href="https://colab.research.google.com/github/jppeirce/DSC210-Foundations-of-Data-Science/blob/main/Homework/HW07-supervised_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 7: How Many Bikes Tomorrow?

**DSC 210 Foundations of Data Science**

*Covers Module 11 (supervised learning: regression).*

```
ASK  ->  GET  ->  EXPLORE  ->  [ MODEL ]  ->  COMMUNICATE
```

References:
- Module 11 course notebook (linear regression, MAE / RMSE / $R^2$, residual plots, k-NN and tree regression, random forests, logistic regression, cross-validation)
- Bike sharing data: Fanaee-T, H. and Gama, J. (2013). Event labeling combining ensemble detectors and background knowledge. *Progress in Artificial Intelligence*, 1-15. [doi:10.1007/s13748-013-0040-3](https://doi.org/10.1007/s13748-013-0040-3) (`csv_data/bike_sharing/day.csv`, also used in Homework 3)

*Last major revision: 2026-11-08*

## Key Concepts

- Fit a line, and state its slope and intercept in the units of the problem
- Judge a regression with MAE, RMSE, $R^2$, a baseline, and a residual plot
- Compare a line with k-NN, a tree, and a random forest, and read feature importances
- Predict a probability with logistic regression, and find its decision boundary
- Compare models with cross-validation, and recognize when the order of the rows matters

## Instructions

**Total: 50 points.** Point values are listed with each problem.

- Cells marked `# RUN-TOGETHER` are complete; run them and read the output.
- Cells marked `# FILL-IN` contain blanks written as `____`.
- Cells marked `# ADAPT` are for you to write. Each one names a cell in the **Module 11 notebook**. Find that cell, copy it here, and change it to fit this problem (a different dataset, different columns, different names). Use the variable names the problem asks for, because later cells use them.
- Run the cells **in order**. Every figure must include a title and labeled axes.
- Most of the credit rides on your written answers. Quote specific numbers from your output.
- Submit the completed `.ipynb`, with all cells run and all figures visible, to Canvas.

---
## Part 0. Back at Capital Bikeshare (2 points)
---

In Homework 3 you graphed two years of daily records from Capital Bikeshare in Washington, D.C. Now your manager wants **predictions**: how many bikes will be rented on a given day? Each extra bike on the street costs money, and each customer who finds an empty dock is lost.

You will use these columns (the setup cell converts the first three back into everyday units):

| Column | Meaning |
| --- | --- |
| `temp_c` | temperature (degrees C) |
| `hum_pct` | humidity (%) |
| `wind_kmh` | wind speed (km/h) |
| `yr` | year: 0 for 2011, 1 for 2012 |
| `workingday` | 1 if the day is neither a weekend nor a holiday |
| `weathersit` | 1 clear, 2 mist or clouds, 3 light rain or snow |
| `dteday` | the date |
| `cnt` | **the label:** total bikes rented that day |

In [ ]:
# RUN-TOGETHER  (setup)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

bikes = pd.read_csv('https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/csv_data/bike_sharing/day.csv')
bikes['temp_c'] = bikes['temp'] * 41          # the file stores temperature divided by 41
bikes['hum_pct'] = bikes['hum'] * 100
bikes['wind_kmh'] = bikes['windspeed'] * 67

print('shape:', bikes.shape)
bikes[['dteday', 'temp_c', 'hum_pct', 'wind_kmh', 'yr', 'workingday', 'weathersit', 'cnt']].describe().round(1)

**Problem 0.** (2 points)

(a) What does one row of `bikes` represent, and how many are there?

(b) What is the mean number of bikes rented per day? Module 11 says a regression must beat the baseline of "always predict the mean." In one sentence, what would that baseline tell the manager each morning?

**Your answer:**

(a)

(b)

---
## Part A. A line through the data (12 points)
---

### Problem A1. Fit the line (3 points)

**How many more bikes are rented on a warmer day?** Fit a line predicting `cnt` from `temp_c`.

In [ ]:
# ADAPT  (Problem A1)
# Module 11, Section 2.2: the RUN-TOGETHER cell that fits `lin` and prints the slope and intercept.
# Use X = bikes[['temp_c']] and y = bikes['cnt'], with test_size=0.25 and random_state=210.
# Keep the names X_train, X_test, y_train, y_test, lin, and pred.
# Print the slope (bikes per degree C), the intercept, and the observed range of temp_c.

(a) Finish the sentence: *"Each additional degree C is associated with ______ more bikes rented per day."*

(b) What does the intercept describe? The coldest day in the data was about 2.4 degrees C. Is the intercept a safe prediction, a mild extrapolation, or a wild one?

(c) The line predicts about 7,680 bikes on a 40 degree C day. Name the problem with trusting that number. (You will see in A3 whether the hottest days really are the busiest.)

**Your answer:**

(a)

(b)

(c)

### Problem A2. How good is the line? (3 points)

In [ ]:
# ADAPT  (Problem A2)
# Module 11, Section 3: the RUN-TOGETHER cell that prints MAE, RMSE, R2, and the baseline MAE.
# Change the units in the printed labels from mpg to bikes.

(a) Report MAE, RMSE, $R^2$, and the baseline MAE. By how many bikes per day does the line improve on the baseline?

(b) Module 11 found $R^2 = 0.700$ for mpg from weight. What does this line's $R^2$ say about how much of the day-to-day variation temperature alone explains?

**Your answer:**

(a)

(b)

### Problem A3. Read the residuals (3 points)

In [ ]:
# ADAPT  (Problem A3)
# Module 11, Section 3: the RUN-TOGETHER cell that draws the residual plot.
# Change the axis labels and title to fit bikes.

In [ ]:
# RUN-TOGETHER  (Problem A3): average rentals in temperature bands, all 731 days
bands = pd.cut(bikes['temp_c'], [0, 10, 15, 20, 25, 28, 31, 36])
bikes.groupby(bands, observed=True)['cnt'].agg(['mean', 'count']).round(0)

(a) Which of the three residual patterns in Module 11, Section 3 does your plot most resemble: no pattern, a curve, or a fan?

(b) Use the temperature-band table to describe the true relationship between temperature and rentals. Where does a straight line go wrong?

**Your answer:**

(a)

(b)

### Problem A4. Errors by hand (3 points)

Use the rounded line $\hat{y} = 1155 + 163x$ for these four days.

| date | temperature $x$ (degrees C) | actual bikes | predicted | residual |
| --- | --- | --- | --- | --- |
| 2011-01-22 | 2.4 | 981 |  |  |
| 2012-07-04 | 32.4 | 7,403 |  |  |
| 2012-09-15 | 24.9 | 8,714 |  |  |
| 2012-10-29 | 18.0 | 22 |  |  |

(a) Complete the table. Then compute the MAE and the RMSE of these four residuals. (Use Python as a calculator if you like.)

(b) October 29, 2012 fell during Hurricane Sandy; only 22 bikes were rented that day. What share of the **sum of absolute residuals** comes from that one day? What share of the **sum of squared residuals**? Which measure, MAE or RMSE, does one huge miss influence more, and why?

**Your answer:**

(a)

(b)

In [ ]:
# Problem A4: your calculator cell (optional)

---
## Part B. Beyond a straight line (12 points)
---

### Problem B1. Let the model bend (4 points)

A3 showed the relationship is curved. Models that do not assume a line might do better.

In [ ]:
# ADAPT  (Problem B1)
# Part 1: Module 11, Section 4: the cell that fits KNeighborsRegressor for several k.
#         Use k = 1, 5, 25, 75. Keep the scaler step, and report MAE in bikes and R2.
# Part 2: Module 11, Section 5.1: the cell that fits DecisionTreeRegressor for several depths.
#         Use max_depth = 1, 3, None, and print train R2 and test R2.

(a) Which k-NN models beat the line on test $R^2$, and by how much? In Module 11 (mpg from weight), k-NN at best only tied the line. Use A3 to explain why bending helps here.

(b) The unlimited tree has the best **training** $R^2$ and the worst **test** $R^2$. Name what happened.

**Your answer:**

(a)

(b)

### Problem B2. More features (4 points)

This is Activity 11.2 with the bike data.

In [ ]:
# FILL-IN  (Problem B2)
FEATURES = ['temp_c', 'hum_pct', 'wind_kmh', 'yr', 'workingday', 'weathersit']

for label, cols in [('one feature (temperature)', ['temp_c']),
                    ('six features', ____)]:
    Xa, Xb, ya, yb = train_test_split(bikes[cols], bikes['cnt'], test_size=0.25, random_state=210)
    print(f'--- {label} ---')
    for name, model in [('linear', LinearRegression()),
                        ('tree, depth 3', DecisionTreeRegressor(max_depth=3, random_state=0)),
                        ('tree, unlimited', DecisionTreeRegressor(random_state=0)),
                        ('random forest', RandomForestRegressor(n_estimators=____, random_state=0))]:
        model.fit(Xa, ya)
        print(f'  {name:17} train R2={r2_score(ya, model.predict(Xa)):.3f}   '
              f'test R2={r2_score(yb, model.____(Xb)):.3f}')
    print()

Use a forest of 200 trees.

(a) With six features, rank the four models by test $R^2$. How much did the best model gain over its one-feature version?

(b) The unlimited tree scores 1.000 on its training days. Is it the best model on the test days? What does the forest do with trees like it?

**Your answer:**

(a)

(b)

### Problem B3. What did the forest lean on? (4 points)

Temperature appears in the file twice: `temp` (actual) and `atemp` ("feels like"). Add `atemp` to the six features and see how the forest shares the credit.

In [ ]:
# ADAPT  (Problem B3)
# Module 11, Section 5.3: the RUN-TOGETHER cell that fits `rf` on all rows and plots feature importance.
# Use the features FEATURES + ['atemp'] and the label bikes['cnt'].

In [ ]:
# RUN-TOGETHER  (Problem B3)
print('correlation of temp and atemp:', round(bikes['temp'].corr(bikes['atemp']), 3))
print()
print('mean bikes per day by year:')
print(bikes.groupby('yr')['cnt'].mean().round(0))

(a) Which feature is second most important? Use the second cell to explain what that feature is really capturing. Is it about the weather?

(b) Without `atemp`, `temp_c` has an importance of about 0.52. With it, `temp_c` and `atemp` share the credit. Use the correlation to explain why, and what this means for reading importances (Module 11, Section 5.3, Discuss).

**Your answer:**

(a)

(b)

---
## Part C. Will it be a busy day? (8 points)
---

The manager plans extra staff when more than 5,000 bikes are rented. Call such a day **busy** (1) and any other day **not busy** (0).

### Problem C1. Fit a logistic regression (4 points)

In [ ]:
# ADAPT  (Problem C1)
# Module 11, Section 6.2: the RUN-TOGETHER cell that fits `log` on the Titanic data.
# Changes:
#   - the label is busy = (bikes['cnt'] > 5000).astype(int)
#   - the features are ['temp_c', 'yr']
#   - do NOT scale the features this time, so the coefficient on temp_c is per degree C
#   - keep test_size=0.25, random_state=210, and stratify on the label
#   - print the test accuracy, the baseline (always predict "not busy"), the intercept, and the two coefficients

(a) Report the test accuracy and the baseline. How much does the model add?

(b) Both coefficients are positive. Say in words what each sign means for the chance of a busy day.

**Your answer:**

(a)

(b)

### Problem C2. Probabilities and the boundary, by hand (4 points)

Round the model to $z = -6.8 + 0.2\,x + 3.6\,\text{yr}$, where $x$ is the temperature in degrees C, and $p = \dfrac{1}{1 + e^{-z}}$. *(You will need $e^{2.8} \approx 16.44$ and $e^{-0.8} \approx 0.449$.)*

| day | $x$ | yr | $z$ | $p$ (busy) | predicted at 0.5 |
| --- | --- | --- | --- | --- | --- |
| a 20 degree day in 2011 | 20 | 0 |  |  |  |
| a 20 degree day in 2012 | 20 | 1 |  |  |  |

(a) Complete the table.

(b) Find the **decision boundary** temperature for each year: solve $z = 0$ for $x$ with yr = 0, then with yr = 1. Compare each boundary with the temperature range in the data (2.4 to 35.3 degrees C). What does the 2011 boundary say about busy days that year?

(c) The table in A3 showed that the very hottest days are *less* busy. Can this logistic model ever predict a lower probability for a hotter day in the same year? Explain from the formula.

**Your answer:**

(a)

(b)

(c)

---
## Part D. How much should we trust these scores? (10 points)
---

### Problem D1. Folds in date order (5 points)

In [ ]:
# ADAPT  (Problem D1)
# Module 11, Activity 11.4: the cell that prints each fold's range and then re-runs with shuffle=True.
# Changes:
#   - print each fold's first and last date (bikes['dteday']) instead of model_year
#   - before shuffling, also print cross_val_score(LinearRegression(), X, y, cv=5, scoring='r2'),
#     the unshuffled scores (Module 11, Section 7.1)
#   - X and y are still the temperature-only data from Problem A1

(a) The rows are in date order. Which months of which year does fold 1 cover? Its $R^2$ is far below zero. Using B3(a), explain why a line trained on the other four folds predicts that fold so badly. (Hint: which year do most of the *training* days come from, and what was the typical temperature during fold 1?)

(b) With `shuffle=True`, report the five scores, their mean, and their spread.

(c) Module 11 says there is one situation where you should **not** shuffle. Your manager wants to use 2011 and 2012 to forecast the days of **2013**. Should the cross-validation for that job be shuffled? Explain.

**Your answer:**

(a)

(b)

(c)

### Problem D2. Choose a model honestly (5 points)

Module 11, Section 7.2 compares models with cross-validation. Here, keep a **test set** aside, run the cross-validation on the **training** days only, choose one model, and only then score it once on the test set.

In [ ]:
# ADAPT  (Problem D2)
# Module 11, Section 7.2: the RUN-TOGETHER cell that compares three models with cross_val_score.
# Changes:
#   - first split: X6_train, X6_test, y6_train, y6_test = train_test_split(bikes[FEATURES], bikes['cnt'],
#                                                                         test_size=0.25, random_state=210)
#   - run cross_val_score on X6_train and y6_train only (not on all the rows)
#   - then fit only the model you choose on the training days, name it `final`, and print its R2 and MAE on X6_test

(a) Which model has the best mean cross-validated $R^2$? Is the gap between the top two larger than the spread of their fold scores?

(b) Report your chosen model's test $R^2$ and MAE. Why is this single test number more trustworthy than if you had tried all three models on the test set and reported the best?

**Your answer:**

(a)

(b)

---
## Part E. Review and report (6 points)
---

### Problem E1. Code review: four quiet bugs (4 points)

A teammate wrote the cell below. **Every line runs.** For each numbered line, say what is wrong and how the output could give it away, then write a corrected version.

In [ ]:
# RUN-TOGETHER  (a teammate's code: it runs, but every line has a problem)

# 1. R2 of the temperature line
print('1. R2:', round(r2_score(pred, y_test), 3))

# 2. RMSE of the temperature line
print('2. RMSE:', round(mean_squared_error(y_test, pred)), 'bikes')

# 3. How many more bikes per degree C?
lin_3 = LinearRegression().fit(bikes.loc[X_train.index, ['temp']], y_train)
print('3. bikes per degree C:', round(lin_3.coef_[0], 1))

# 4. How well will the forest do on new days?
print('4. forest R2 on new days:', round(r2_score(y6_train, final.predict(X6_train)), 3))

**Your answer:**

| Line | What is wrong | How the output gives it away |
| --- | --- | --- |
| 1 |  |  |
| 2 |  |  |
| 3 |  |  |
| 4 |  |  |

In [ ]:
# Problem E1: your corrected lines here (label each one 1 to 4)

### Problem E2. Report to the manager (2 points)

In 3 to 4 sentences for a manager who has never fit a model: which model would you use to plan bikes for tomorrow, how far off should she expect it to be on a typical day (in bikes), and one caution about using it next year.

**Your answer:**